# Relax Inc. | Which new users become adopted users?


In [3]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

## Identify three visits on distinct days in a rolling week

In [4]:
users = pd.read_csv("data/takehome_users.csv", encoding="latin1", parse_dates=["creation_time"]).set_index("object_id")
events = pd.read_csv("data/takehome_user_engagement.csv", parse_dates=["time_stamp"])
assert users.index.is_unique and events.user_id.isin(users.index).all()
days = (events.loc[events.visited.eq(1)].assign(day=lambda d: d.time_stamp.dt.normalize())
        [["user_id", "day"]].drop_duplicates().sort_values(["user_id", "day"]))
# Three distinct dates fit in seven calendar dates when first-to-third span <= 6.
span = days.day - days.groupby("user_id").day.shift(2)
first_adoption = days[span.le(pd.Timedelta(days=6))].groupby("user_id").day.min()
users["adopted_on"] = first_adoption.reindex(users.index)
users["adopted"] = users.adopted_on.notna()
print("Ever adopted", users.adopted.sum(), len(users), users.adopted.mean())
print(users.groupby("creation_source").adopted.agg(["size", "mean"]).round(4))

Ever adopted 1602 12000 0.1335
                    size    mean
creation_source                 
GUEST_INVITE        2163  0.1664
ORG_INVITE          4254  0.1300
PERSONAL_PROJECTS   2111  0.0777
SIGNUP              2087  0.1404
SIGNUP_GOOGLE_AUTH  1385  0.1675


## Build a time-based evaluation set

In [5]:
# Equal follow-up for prediction. Freeze training labels before test signups begin.
signup = users.creation_time.dt.normalize()
last_day = days.day.max()
eligible = signup + pd.Timedelta(days=29) <= last_day
y = users.adopted_on.le(signup + pd.Timedelta(days=29))
split = pd.Timestamp("2014-02-01")
train = signup + pd.Timedelta(days=29) < split
test = eligible & signup.ge(split)
terms = ["creation_source", "opted_in_to_mailing_list", "enabled_for_marketing_drip"]

## Compare signup source and marketing flags

In [6]:
def evaluate(features):
    model = make_pipeline(ColumnTransformer([("encode", OneHotEncoder(handle_unknown="ignore"), features)]),
                          LogisticRegression(max_iter=500))
    model.fit(users.loc[train, features], y[train])
    scores = model.predict_proba(users.loc[test, features])[:, 1]
    print(features, "ROC AUC", roc_auc_score(y[test], scores),
          "AP", average_precision_score(y[test], scores))
print("Test count, positive rate, train count", test.sum(), y[test].mean(), train.sum())
evaluate(terms[:1])
evaluate(terms)
print("30-day adoption, held-out signups")
print(users.loc[test, ["creation_source"]].assign(adopted=y[test]).groupby("creation_source").adopted.agg(["size", "mean"]).round(4))

Test count, positive rate, train count 2177 0.03307303628847037 8510
['creation_source'] ROC AUC 0.561797307996833 AP 0.03926735073916324
['creation_source', 'opted_in_to_mailing_list', 'enabled_for_marketing_drip'] ROC AUC 0.5182139086830297 AP 0.03660422939913384
30-day adoption, held-out signups
                    size    mean
creation_source                 
GUEST_INVITE         386  0.0466
ORG_INVITE           803  0.0349
PERSONAL_PROJECTS    341  0.0147
SIGNUP               390  0.0359
SIGNUP_GOOGLE_AUTH   257  0.0272


## Check sensitivity to the adoption window

In [7]:
# Also evaluate the exact "ever observed" target. At the simulated fit date
# training labels may only use adoption events already observed by that date.
y = users.adopted.copy()
y.loc[train] = users.loc[train, "adopted_on"].lt(split)
print("Ever observed on held-out signups: base rate", y[test].mean())
evaluate(terms[:1])
evaluate(terms)

Ever observed on held-out signups: base rate 0.09003215434083602
['creation_source'] ROC AUC 0.5467747169539194 AP 0.09785666514050881
['creation_source', 'opted_in_to_mailing_list', 'enabled_for_marketing_drip'] ROC AUC 0.5333280965086691 AP 0.09878207346860181


**Forward prediction.** We tested signup-source logistic regression on 2,177 users registered from February 1, 2014 onward with at least 30 observable calendar dates. Training used 8,510 earlier signups and only labels available before February 1. The observed-adoption target gives later users less follow-up, so we also tested adoption within the first 30 calendar dates for everyone.

| Target on later signups | Test prevalence | Source-only ROC AUC | Source-only average precision | Average precision with marketing flags |
|---|---:|---:|---:|---:|
| Ever adopted by end of observation | 9.00% | 0.547 | 0.098 | 0.099 |
| Adopted in first 30 calendar dates | 3.31% | 0.562 | 0.039 | 0.037 |

Prevalence is the constant-score baseline for average precision. Marketing opt-in and drip enrollment added little, even assuming those snapshot fields were available at signup. Last-session time would reveal later activity, so we excluded it. Personal workspace invitees also had the lowest 30-day rate on the holdout (1.47%). Google signups’ stronger full-period rate did not carry through to that early outcome (2.72%).

**Recommendation.** Test an onboarding prompt that gives personal-workspace invitees a specific shared task to return to. Randomize by workspace to limit spillover and compare 30-day adoption. Capture first collaboration and workspace activity at invitation time to investigate whether the gap reflects onboarding friction or how active the inviting team already was.